# Gold Dimensions

Build the shared dimensions used across the Gold fact tables.

Dimensions:
- `dim_date`
- `dim_patient`
- `dim_careunit`
- `dim_service`
- `dim_diagnosis`
- `dim_procedure`

The Gold layer uses cleaned Silver data directly. Row-level cleaning and standardisation already completed in Silver are not repeated here.

In [0]:
from pyspark.sql import functions as F

# Unity Catalog locations
SILVER_SCHEMA = "cdr_silver.processed_mimic_iv"
GOLD_SCHEMA = "cdr_gold.analytics_mimic_iv"

# Ensure Gold schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")

print(f"Silver source: {SILVER_SCHEMA}")
print(f"Gold target:   {GOLD_SCHEMA}")

In [0]:
# Load only Silver tables required for dimensions

patients = spark.table(f"{SILVER_SCHEMA}.patients")
admissions = spark.table(f"{SILVER_SCHEMA}.admissions")
transfers = spark.table(f"{SILVER_SCHEMA}.transfers")
services = spark.table(f"{SILVER_SCHEMA}.services")
icustays = spark.table(f"{SILVER_SCHEMA}.icustays")

diagnosis_reference = spark.table(
    f"{SILVER_SCHEMA}.d_icd_diagnoses"
)

procedure_reference = spark.table(
    f"{SILVER_SCHEMA}.d_icd_procedures"
)

print("Dimension source tables loaded.")

In [0]:
# Build patient dimension
# Grain: one row per subject_id

dim_patient = (
    patients
    .select(
        "subject_id",
        "gender"
    )
    .dropDuplicates(["subject_id"])
    .withColumn(
        "patient_key",
        F.xxhash64(F.col("subject_id")).cast("long")
    )
    .select(
        "patient_key",
        "subject_id",
        "gender"
    )
)

# Basic validation
patient_validation = dim_patient.agg(
    F.count("*").alias("rows"),
    F.countDistinct("subject_id").alias("unique_patients"),
    F.countDistinct("patient_key").alias("unique_patient_keys"),
    F.sum(
        F.when(F.col("patient_key").isNull(), 1).otherwise(0)
    ).alias("null_keys")
)

display(patient_validation)

In [0]:
# Build date dimension
# Grain: one row per calendar date

date_bounds = (
    admissions
    .select(
        F.min(F.to_date("admittime")).alias("min_date"),
        F.max(F.to_date("dischtime")).alias("max_date")
    )
    .first()
)

min_date = date_bounds["min_date"]
max_date = date_bounds["max_date"]

dim_date = (
    spark.sql(
        f"""
        SELECT explode(
            sequence(
                to_date('{min_date}'),
                to_date('{max_date}'),
                interval 1 day
            )
        ) AS calendar_date
        """
    )
    .withColumn(
        "date_key",
        F.date_format("calendar_date", "yyyyMMdd").cast("int")
    )
    .withColumn("year", F.year("calendar_date"))
    .withColumn("month", F.month("calendar_date"))
    .withColumn("month_name", F.date_format("calendar_date", "MMMM"))
    .withColumn("quarter", F.quarter("calendar_date"))
    .withColumn("week_of_year", F.weekofyear("calendar_date"))
    .withColumn("day_of_month", F.dayofmonth("calendar_date"))
    .withColumn("day_of_week", F.date_format("calendar_date", "EEEE"))
    .withColumn(
        "is_weekend",
        F.dayofweek("calendar_date").isin(1, 7)
    )
    .select(
        "date_key",
        "calendar_date",
        "year",
        "quarter",
        "month",
        "month_name",
        "week_of_year",
        "day_of_month",
        "day_of_week",
        "is_weekend"
    )
)

# Basic validation
date_validation = dim_date.agg(
    F.count("*").alias("rows"),
    F.countDistinct("date_key").alias("unique_date_keys"),
    F.min("calendar_date").alias("min_date"),
    F.max("calendar_date").alias("max_date"),
    F.sum(
        F.when(F.col("date_key").isNull(), 1).otherwise(0)
    ).alias("null_keys")
)

display(date_validation)

In [0]:
# Build careunit dimension
# Grain: one row per standardised careunit

# ICU careunits identified from ICU stay records
icu_careunits = (
    icustays
    .select(F.col("first_careunit").alias("careunit_name"))
    .union(
        icustays.select(F.col("last_careunit").alias("careunit_name"))
    )
    .filter(F.col("careunit_name").isNotNull())
    .dropDuplicates()
    .withColumn("is_icu", F.lit(True))
)

# All careunits observed in hospital transfers
all_careunits = (
    transfers
    .select(F.col("careunit").alias("careunit_name"))
    .filter(F.col("careunit_name").isNotNull())
    .dropDuplicates()
)

dim_careunit = (
    all_careunits
    .join(
        icu_careunits,
        on="careunit_name",
        how="left"
    )
    .withColumn(
        "is_icu",
        F.coalesce(F.col("is_icu"), F.lit(False))
    )
    .withColumn(
        "careunit_group",
        F.when(F.col("careunit_name") == "UNKNOWN", "Unknown")
         .when(F.col("is_icu"), "ICU")
         .otherwise("Non-ICU")
    )
    .withColumn(
        "careunit_key",
        F.xxhash64(F.col("careunit_name")).cast("long")
    )
    .select(
        "careunit_key",
        "careunit_name",
        "careunit_group",
        "is_icu"
    )
)

# Basic validation
careunit_validation = dim_careunit.agg(
    F.count("*").alias("rows"),
    F.countDistinct("careunit_name").alias("unique_careunits"),
    F.countDistinct("careunit_key").alias("unique_careunit_keys"),
    F.sum(
        F.when(F.col("careunit_key").isNull(), 1).otherwise(0)
    ).alias("null_keys"),
    F.sum(
        F.when(F.col("is_icu"), 1).otherwise(0)
    ).alias("icu_careunits")
)

display(careunit_validation)

In [0]:
# Build service dimension
# Grain: one row per hospital service code

service_codes = (
    services
    .select(F.col("curr_service").alias("service_code"))
    .union(
        services.select(F.col("prev_service").alias("service_code"))
    )
    .filter(F.col("service_code").isNotNull())
    .dropDuplicates()
)

dim_service = (
    service_codes
    .withColumn(
        "service_key",
        F.xxhash64(F.col("service_code")).cast("long")
    )
    .select(
        "service_key",
        "service_code"
    )
)

# Basic validation
service_validation = dim_service.agg(
    F.count("*").alias("rows"),
    F.countDistinct("service_code").alias("unique_services"),
    F.countDistinct("service_key").alias("unique_service_keys"),
    F.sum(
        F.when(F.col("service_key").isNull(), 1).otherwise(0)
    ).alias("null_keys")
)

display(service_validation)

In [0]:
# Build diagnosis dimension
# Grain: one row per ICD code + ICD version

dim_diagnosis = (
    diagnosis_reference
    .select(
        "icd_code",
        "icd_version",
        "description"
    )
    .dropDuplicates([
        "icd_code",
        "icd_version"
    ])
    .withColumn(
        "diagnosis_key",
        F.xxhash64(
            F.col("icd_code"),
            F.col("icd_version")
        ).cast("long")
    )
    .select(
        "diagnosis_key",
        "icd_code",
        "icd_version",
        "description"
    )
)

# Basic validation
diagnosis_validation = dim_diagnosis.agg(
    F.count("*").alias("rows"),
    F.countDistinct(
        F.struct("icd_code", "icd_version")
    ).alias("unique_diagnoses"),
    F.countDistinct("diagnosis_key").alias("unique_diagnosis_keys"),
    F.sum(
        F.when(F.col("diagnosis_key").isNull(), 1).otherwise(0)
    ).alias("null_keys"),
    F.sum(
        F.when(F.col("description").isNull(), 1).otherwise(0)
    ).alias("missing_descriptions")
)

display(diagnosis_validation)

In [0]:
# Build procedure dimension
# Grain: one row per ICD procedure code + ICD version

dim_procedure = (
    procedure_reference
    .select(
        "icd_code",
        "icd_version",
        "description"
    )
    .dropDuplicates([
        "icd_code",
        "icd_version"
    ])
    .withColumn(
        "procedure_key",
        F.xxhash64(
            F.col("icd_code"),
            F.col("icd_version")
        ).cast("long")
    )
    .select(
        "procedure_key",
        "icd_code",
        "icd_version",
        "description"
    )
)

# Basic validation
procedure_validation = dim_procedure.agg(
    F.count("*").alias("rows"),
    F.countDistinct(
        F.struct("icd_code", "icd_version")
    ).alias("unique_procedures"),
    F.countDistinct("procedure_key").alias("unique_procedure_keys"),
    F.sum(
        F.when(F.col("procedure_key").isNull(), 1).otherwise(0)
    ).alias("null_keys"),
    F.sum(
        F.when(F.col("description").isNull(), 1).otherwise(0)
    ).alias("missing_descriptions")
)

display(procedure_validation)

In [0]:
# Publish shared dimensions to Gold
# Dimensions are small, so a full overwrite is simple and reliable.

dimension_tables = {
    "dim_patient": dim_patient,
    "dim_date": dim_date,
    "dim_careunit": dim_careunit,
    "dim_service": dim_service,
    "dim_diagnosis": dim_diagnosis,
    "dim_procedure": dim_procedure
}

for table_name, df in dimension_tables.items():

    output_df = df.withColumn(
        "_gold_timestamp",
        F.current_timestamp()
    )

    (
        output_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{GOLD_SCHEMA}.{table_name}")
    )

    print(f"Published: {GOLD_SCHEMA}.{table_name}")

In [0]:
# Validate persisted Gold dimensions

dimension_keys = {
    "dim_patient": "patient_key",
    "dim_date": "date_key",
    "dim_careunit": "careunit_key",
    "dim_service": "service_key",
    "dim_diagnosis": "diagnosis_key",
    "dim_procedure": "procedure_key"
}

validation_rows = []

for table_name, key_column in dimension_keys.items():

    df = spark.table(f"{GOLD_SCHEMA}.{table_name}")

    result = (
        df.agg(
            F.count("*").alias("row_count"),
            F.countDistinct(key_column).alias("unique_keys"),
            F.sum(
                F.when(F.col(key_column).isNull(), 1).otherwise(0)
            ).alias("null_keys")
        )
        .first()
    )

    validation_rows.append({
        "table": table_name,
        "row_count": result["row_count"],
        "unique_keys": result["unique_keys"],
        "null_keys": result["null_keys"],
        "status": (
            "PASS"
            if result["row_count"] == result["unique_keys"]
            and result["null_keys"] == 0
            else "CHECK"
        )
    })

display(
    spark.createDataFrame(validation_rows)
    .orderBy("table")
)